In [ ]:
from cellpose import models, io
from bioio import BioImage
import napari
from skimage.transform import rescale
from skimage.io import imsave
import random
from glob import glob

# Testing parameters for cellpose 4

In [ ]:
test_img = r"E:\KristinG\20260625_zebrafish\fin\WT_ctrl_2_skin_3_dorsal_fin_Out.czi"
img = BioImage(test_img)
img_rd = img.get_image_data("ZYX", C=1)

In [ ]:
img_twoch = img.get_image_data("ZYXC")

In [ ]:
img_twoch.shape

In [ ]:
cp = io.logger_setup()
model = models.CellposeModel(gpu=True)

In [ ]:
rescaled_img = rescale(img_rd, [0.5,0.2,0.2], anti_aliasing=True, channel_axis=None,preserve_range=True)

In [ ]:
new_scale = img.physical_pixel_sizes[0]*0.5, img.physical_pixel_sizes[1]*0.2, img.physical_pixel_sizes[2]*0.2

In [ ]:
anisotropy = new_scale[0]/new_scale[1]

In [ ]:
rescaled_masks_stitch, rescaled_flows_stitch, _ = model.eval(rescaled_img, z_axis=0, batch_size=32, do_3D=False,normalize=True,stitch_threshold=0.7,anisotropy=anisotropy)

In [ ]:
v = napari.Viewer()

In [ ]:
v.add_image(rescaled_img, name='rescaled image', scale=new_scale,channel_axis=None)
v.add_labels(rescaled_masks_stitch, name='rescaled masks stitch 0.7 norm anisotropy', opacity=0.5,scale=new_scale)

In [ ]:
props = regionprops_table(rescaled_masks_stitch, properties=['label'])

In [ ]:
#v2.add_image(rescaled_img, name='rescaled image', scale=(img.physical_pixel_sizes[0]*(200/30), img.physical_pixel_sizes[1]*(200/30), img.physical_pixel_sizes[2]*(200/30)))
random_subset = []
while len(random_subset) < 10:
    r = randint(1, len(props['label']))
    if r not in random_subset:
        random_subset.append(r)

for i in random_subset:
    l = props['label'][i]
    obj = rescaled_masks_stitch == l
    v.add_labels(obj, name=f'object {l}', scale=new_scale, opacity=0.5)

In [ ]:
for l in props['equivalent_diameter_area']:
    print(l)

In [ ]:
save = r"E:\KristinG\20260625_zebrafish\fin_predictions"
imsave(r"E:\KristinG\20260625_zebrafish\fin_rescaled\DKO_ctrl_1_skin_1_ventral_fin_Out_rescaled.tif", rescaled_img.astype('uint16'))
imsave(r"E:\KristinG\20260625_zebrafish\fin_predictions\DKO_ctrl_1_skin_1_ventral_fin_Out_rescaled_masks_stitch.tif", rescaled_masks_stitch.astype('uint16'))

# Run cellpose4 on random subset of imgs

In [ ]:
img_files = sorted(glob(r"E:\KristinG\20260625_zebrafish\fin\*.czi"))
wt_files = [f for f in img_files if "WT" in f]
dko_files = [f for f in img_files if "DKO" in f]
rand_WT = random.sample(wt_files, 5)
rand_DKO = random.sample(dko_files, 5)

In [ ]:
wt_imgs = [BioImage(f).get_image_data("ZYX", C=1) for f in rand_WT]
dko_imgs = [BioImage(f).get_image_data("ZYX", C=1) for f in rand_DKO]
rescaled_wt_imgs = [rescale(img, [0.5,0.2,0.2], anti_aliasing=True, channel_axis=None,preserve_range=True) for img in wt_imgs]
rescaled_dko_imgs = [rescale(img, [0.5,0.2,0.2], anti_aliasing=True, channel_axis=None,preserve_range=True) for img in dko_imgs]
del(wt_imgs, dko_imgs)

In [ ]:
io.logger_setup()


In [ ]:
model = models.CellposeModel(gpu=True)

In [ ]:
all_imgs = rescaled_wt_imgs + rescaled_dko_imgs

In [ ]:
masks, _, _ = model.eval(all_imgs, batch_size=2, z_axis=0, do_3D=False,normalize=True,stitch_threshold=0.7)

In [ ]:
for i, img in enumerate(zip(masks, all_imgs)):
    m, img = img
    save = r"E:\KristinG\20260625_zebrafish\fin_predictions"
    imsave(rf"E:\KristinG\20260625_zebrafish\fin_predictions\{i}_rescaled_masks_stitch.tif", m.astype('uint16'))
    imsave(rf"E:\KristinG\20260625_zebrafish\fin_rescaled\{i}_rescaled.tif", img.astype('uint16'))